In [1]:
# -*- coding: utf-8 -*-
"""
Forced PDP/ICE plotting for subjectively malodorous odor descriptors.

目的：
1. 对主观上可视为恶臭/刺激性/环境异味的 odor descriptors 都绘制代表性交互图；
2. 即使某些标签没有通过 useful_for_plot 筛选，也强制画至少 1 组代表性 PDP/ICE；
3. 对每个标签优先选择 useful interaction；如果没有 useful，则选择得分最高的代表性 pair；
4. 输出：
   - 2D PDP probability heatmap / contour
   - PDP interaction residual heatmap / contour
   - ICE curves
   - 所有候选 pair 筛选表
   - 最终绘图 pair 汇总表
5. 图片只保存 PNG；
6. dpi = 800；
7. 蓝白配色，无标题，Arial 优先。
"""

import os
import re
import json
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager as fm
import xgboost as xgb

warnings.filterwarnings("ignore")


# ============================================================
# 0. 用户配置
# ============================================================

RANDOM_SEED = 42

FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"
SHEET_NAME = 0

OUT_DIR = "./Forced_subjective_malodor_PDP_ICE_plots"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
MODEL_DIR = os.path.join(OUT_DIR, "models")

for d in [OUT_DIR, PLOT_DIR, TABLE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# 只保存 PNG，dpi=800
DPI = 800

SAVE_PDF = False
SAVE_SVG = False

# 是否绘制 ICE 曲线
PLOT_ICE = True

# 每个主观恶臭标签至少绘制几个 pair
MIN_PLOTS_PER_LABEL = 1

# 每个主观恶臭标签最多绘制几个 pair
MAX_PLOTS_PER_LABEL = 2

# 候选筛选数量
TOPK_FEATURES_PER_GROUP = 8
MAX_PAIRS_PER_GROUP_COMBO = 80

# PDP / ICE 样本量
MAX_SCREEN_PDP_SAMPLES = 700
MAX_FINAL_PDP_SAMPLES = 1500
MAX_ICE_SAMPLES = 300

# 连续变量网格点数
N_GRID_CONTINUOUS = 30

# 唯一值数量 <= 该值时按离散变量处理
MAX_UNIQUE_AS_DISCRETE = 6

# 支持度阈值
MIN_POSITIVES_PER_LABEL = 10
MIN_FEATURE_SUPPORT_BINARY = 20
MIN_PAIR_COOC_SUPPORT_BINARY = 6

# useful interaction 判断阈值
MIN_PDP_RANGE = 0.020
MIN_MEAN_ABS_RESIDUAL = 0.003
MIN_MAX_ABS_RESIDUAL = 0.010
MIN_BINARY_INTERACTION_CONTRAST = 0.020

# 二值特征冗余阈值
MAX_BINARY_PHI_CORR = 0.98
MAX_BINARY_JACCARD = 0.98

# ICE 图设置
MAX_ICE_LINES_PER_LEVEL = 80

# 是否强制对主观恶臭标签出图
FORCE_PLOT_SUBJECTIVE_MALODORS = True


# ============================================================
# 1. 标签定义
# ============================================================

TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

"""
主观恶臭 / 刺激性 / 环境异味描述词。

说明：
- sulfurous = 硫磺味 / 含硫臭味；
- garlic / cabbage / fishy / cheesy / sweaty / musty 都属于常见异味；
- pungent / sharp / sour / burnt / solvent 属于刺激性或不愉悦气味；
- aldehydic / ketonic 在环境 VOC 中常与刺激性/氧化味相关，因此作为补充。
"""
SUBJECTIVE_MALODOR_LABELS = [
    "sulfurous",
    "garlic",
    "cabbage",
    "fishy",
    "pungent",
    "sharp",
    "sour",
    "cheesy",
    "sweaty",
    "musty",
    "burnt",
    "solvent",
    "aldehydic",
    "ketonic",
]

ANALYSIS_LABELS = SUBJECTIVE_MALODOR_LABELS.copy()


# ============================================================
# 2. XGBoost 参数
# ============================================================

BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)


# ============================================================
# 3. 结构组定义
# ============================================================

GROUP_PATTERNS = {
    "Sulfur_groups": [
        r"sulfur", r"sulphur", r"groupscontainingsul", r"containing\s*sul",
        r"atom:\s*s", r"atom\s*count:\s*s", r"atomcount:\s*s",
        r"kg_element_s", r"thiol", r"thio", r"sulfide", r"sulfone",
        r"sulfoxide", r"sulfanyl", r"mercapto", r"disulfide",
        r"trisulfide", r"thiophene", r"\(-sh\)", r"n=c=s",
        r"isothiocyanate"
    ],

    "Amines": [
        r"amine", r"amines", r"amino", r"aniline", r"ammonia",
        r"ammonium", r"imino", r"imine", r"nitrogen",
        r"groupscontainingnitrogen", r"atom:\s*n", r"atom\s*count:\s*n",
        r"atomcount:\s*n", r"kg_element_n", r"pyridine", r"pyrrole",
        r"indole", r"n\s*≥", r"n\s*>=", r"n="
    ],

    "Aldehydes_Carbonyls": [
        r"aldehyde", r"aldehydic", r"formyl", r"carbonyl",
        r"c\(=o\)", r"\[cx3\]\(=o\)", r"\[cx3h1\]\(=o\)",
        r"cc\(=o\)", r"ketone", r"ketonic", r"acrolein",
        r"propanal", r"butanal", r"hexanal", r"benzaldehyde"
    ],

    "Carboxylic_acids": [
        r"carboxylic", r"carboxyl", r"-cooh", r"cooh",
        r"c\(=o\)\[ox2h1\]", r"c\(=o\)ox2h1",
        r"cc\(=o\)o", r"acid", r"fatty", r"acetic",
        r"propionic", r"butyric", r"valeric", r"isovaleric"
    ],

    "Ethers_Esters": [
        r"ether", r"ester", r"ethereal", r"acetal",
        r"lactone", r"acetate", r"c-o-c", r"coc"
    ],

    "Aromatics_Heterocycles": [
        r"aromatic", r"benzene", r"phenyl", r"toluene",
        r"xylene", r"styrene", r"heteroaromatic",
        r"thiophene", r"furan", r"pyridine", r"pyrrole",
        r"aromaticmonocyclic"
    ],

    "Halogens": [
        r"halogen", r"chloro", r"bromo", r"fluoro", r"iodo",
        r"kg_element_cl", r"kg_element_br", r"fg:\s*cl", r"chloride",
        r"trifluoromethyl"
    ],
}


LABEL_GROUP_PAIR_PLAN = {
    "sulfurous": [
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
        ("Sulfur_groups", "Ethers_Esters"),
    ],
    "garlic": [
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
        ("Sulfur_groups", "Ethers_Esters"),
    ],
    "cabbage": [
        ("Sulfur_groups", "Carboxylic_acids"),
        ("Sulfur_groups", "Amines"),
        ("Sulfur_groups", "Aldehydes_Carbonyls"),
    ],
    "fishy": [
        ("Amines", "Sulfur_groups"),
        ("Amines", "Aldehydes_Carbonyls"),
        ("Amines", "Ethers_Esters"),
    ],
    "pungent": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Sulfur_groups", "Amines"),
    ],
    "sharp": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Aldehydes_Carbonyls", "Ethers_Esters"),
    ],
    "sour": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "cheesy": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "sweaty": [
        ("Carboxylic_acids", "Sulfur_groups"),
        ("Carboxylic_acids", "Amines"),
        ("Carboxylic_acids", "Aldehydes_Carbonyls"),
    ],
    "musty": [
        ("Ethers_Esters", "Aromatics_Heterocycles"),
        ("Ethers_Esters", "Aldehydes_Carbonyls"),
        ("Aromatics_Heterocycles", "Sulfur_groups"),
    ],
    "burnt": [
        ("Aldehydes_Carbonyls", "Aromatics_Heterocycles"),
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Ethers_Esters", "Aldehydes_Carbonyls"),
    ],
    "solvent": [
        ("Ethers_Esters", "Aromatics_Heterocycles"),
        ("Halogens", "Aromatics_Heterocycles"),
        ("Aldehydes_Carbonyls", "Ethers_Esters"),
    ],
    "aldehydic": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Aldehydes_Carbonyls", "Aromatics_Heterocycles"),
    ],
    "ketonic": [
        ("Aldehydes_Carbonyls", "Sulfur_groups"),
        ("Aldehydes_Carbonyls", "Amines"),
        ("Aldehydes_Carbonyls", "Aromatics_Heterocycles"),
    ],
}


# ============================================================
# 4. 人工优先候选 pair
# ============================================================

MANUAL_PAIRS = {
    "sulfurous": [
        ("KG_Ancestor__GroupsContainingSulfur", "Atom count: N >= 2"),
        ("KG_Ancestor__GroupsContainingSulfur", "FG: [CX3](=O)[#6][#6]"),
        ("KG_Ancestor__GroupsContainingSulfur", "KG_FG__Ether"),
        ("FG: Thiol(–SH)", "KG_Ancestor__Amines"),
        ("FG: thioether-sulfide", "FG: [CX3H1](=O)[#6]"),
    ],

    "garlic": [
        ("KG_Ancestor__GroupsContainingSulfur", "Atom count: N >= 2"),
        ("FG: Disulfide(S–S)", "Atom count: N >= 2"),
        ("KG_Ancestor__GroupsContainingSulfur", "FG: [CX3H1](=O)[#6]"),
    ],

    "cabbage": [
        ("KG_Ancestor__GroupsContainingSulfur", "FG: Carboxylic acid(–COOH)"),
        ("KG_Ancestor__GroupsContainingSulfur", "KG_Ancestor__Amines"),
        ("Atom count: S >= 2", "FG: Disulfide(S–S)"),
    ],

    "fishy": [
        ("KG_Ancestor__GroupsContainingNitrogen", "KG_FG__Ether"),
        ("KG_Ancestor__GroupsContainingNitrogen", "KG_Ancestor__Ether"),
        ("KG_Ancestor__Amines", "KG_FG__Ether"),
        ("KG_Ancestor__Amines", "KG_Ancestor__GroupsContainingSulfur"),
    ],

    "pungent": [
        ("FG: [CX3](=O)[#6][#6]", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: [CX3H1](=O)[#6]", "KG_Ancestor__Amines"),
        ("FG: N=C=S", "FG: [CX3H1](=O)[#6]"),
    ],

    "sharp": [
        ("FG: [CX3H1](=O)[#6]", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: [CX3H1](=O)[#6]", "KG_Ancestor__Amines"),
        ("FG: [CX3](=O)[#6][#6]", "Atom count: S >= 1"),
    ],

    "sour": [
        ("FG: Carboxylic acid(–COOH)", "FG: CC(=O)O"),
        ("FG: Carboxylic acid(–COOH)", "C(=O)[OH] && (NumAliphaticCarbons >= 10)"),
        ("FG: Carboxylic acid(–COOH)", "C(=O)[OH] && (MolWt < 110)"),
        ("FG: Carboxylic acid(–COOH)", "KG_Ancestor__GroupsContainingSulfur"),
    ],

    "cheesy": [
        ("FG: Carboxylic acid(–COOH)", "C(=O)O&&(TPSA>60)&&(LogP<1.0)"),
        ("FG: Carboxylic acid(–COOH)", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: Carboxylic acid(–COOH)", "FG: [CX3](=O)[#6][#6]"),
    ],

    "sweaty": [
        ("FG: Carboxylic acid(–COOH)", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: Carboxylic acid(–COOH)", "KG_Ancestor__Amines"),
        ("C(=O)[OH] && (NumAliphaticCarbons >= 10)", "KG_Ancestor__GroupsContainingSulfur"),
    ],

    "musty": [
        ("KG_FG__Ether", "KG_Ancestor__AromaticMonocyclicSulfurHeterocycle"),
        ("KG_FG__Ether", "FG: [CX3](=O)[#6][#6]"),
        ("KG_FG__Ether", "KG_Ancestor__GroupsContainingSulfur"),
    ],

    "burnt": [
        ("FG: [CX3](=O)[#6][#6]", "KG_Ancestor__AromaticMonocyclicCarbocycle"),
        ("FG: ketone", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG:furan", "KG_FG__Carbonyl"),
    ],

    "solvent": [
        ("KG_FG__Ether", "KG_Ancestor__AromaticMonocyclicCarbocycle"),
        ("FG: Halogen-containing motif (token: F-Cl-Br-I)", "KG_Ancestor__AromaticMonocyclicCarbocycle"),
        ("FG: Ester(–COOR)", "KG_Ancestor__AromaticMonocyclicCarbocycle"),
    ],

    "aldehydic": [
        ("FG: [CX3H1](=O)[#6]", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: [CX3H1](=O)[#6]", "KG_Ancestor__Amines"),
        ("FG: c[CX3H1](=O)", "KG_Ancestor__GroupsContainingSulfur"),
    ],

    "ketonic": [
        ("FG: [CX3](=O)[#6][#6]", "KG_Ancestor__GroupsContainingSulfur"),
        ("FG: [CX3](=O)[#6][#6]", "KG_Ancestor__Amines"),
        ("FG: ketone", "KG_Ancestor__AromaticMonocyclicCarbocycle"),
    ],
}


# ============================================================
# 5. 绘图样式
# ============================================================

FONT_X_LABEL = 16
FONT_Y_LABEL = 16
FONT_TICK = 18
FONT_ANNOT = 20

CBAR_LABEL_FONT_PROB = 14
CBAR_LABEL_FONT_RESID = 18
CBAR_TICK_FONT = 18

AXIS_LINEWIDTH = 2.2
TICK_WIDTH = 2.0
TICK_LENGTH = 6

MEAN_LINE_WIDTH = 3.4
ICE_LINE_WIDTH = 1.0

RESIDUAL_COLOR_BY_ABS = True
ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR = False

BLUE_WHITE_CMAP = LinearSegmentedColormap.from_list(
    "custom_blue_white",
    [
        "#ffffff",
        "#eff6fb",
        "#d9eaf7",
        "#bdd7e7",
        "#6baed6",
        "#3182bd",
        "#08519c",
        "#08306b"
    ]
)

ICE_MEAN_COLORS = ["#08306b", "#2171b5", "#6baed6", "#9ecae1"]
ICE_INDIVIDUAL_COLOR = "#9ecae1"


FEATURE_LABEL_MAP = {
    "FG: Carboxylic acid(–COOH)": "Carboxylic acid (-COOH)",
    "FG: Carboxylic acid(-COOH)": "Carboxylic acid (-COOH)",

    "KG_Ancestor__GroupsContainingNitrogen": "N-containing groups",
    "KG_Ancestor__GroupsContainingSulfur": "S-containing groups",
    "KG_Ancestor__Amines": "Amines",

    "KG_FG__Ether": "Ether",
    "KG_Ancestor__Ether": "Ether-related groups",

    "Atom count: N >= 2": "N atom count",

    "C(=O)O&&(TPSA>60)&&(LogP<1.0)": "C(=O)O && TPSA>60 && LogP<1.0",
    "C(=O)O && (TPSA>60) && (LogP<1.0)": "C(=O)O && TPSA>60 && LogP<1.0",
    "C(=O)O && (TPSA > 60) && (LogP < 1.0)": "C(=O)O && TPSA>60 && LogP<1.0",

    "C(=O)[OH] && (MolWt < 110)": "C(=O)[OH] && MW<110",
    "C(=O)[OH]&&(MolWt<110)": "C(=O)[OH] && MW<110",

    "C(=O)[OH] && (NumAliphaticCarbons >= 10)": "C(=O)[OH] && Aliphatic C≥10",
    "C(=O)[OH]&&(NumAliphaticCarbons>=10)": "C(=O)[OH] && Aliphatic C≥10",

    "FG: CC(=O)O": "CC(=O)O",
    "FG: [CX3](=O)[#6][#6]": "Ketone-like carbonyl",
    "FG: [CX3H1](=O)[#6]": "Aldehyde",
    "FG: c[CX3H1](=O)": "Aromatic aldehyde",
    "FG: ketone": "Ketone",
    "FG:furan": "Furan",
    "KG_FG__Carbonyl": "Carbonyl",
    "FG: Disulfide(S–S)": "Disulfide (S-S)",
    "FG: Thiol(–SH)": "Thiol (-SH)",
    "FG: thioether-sulfide": "Thioether/sulfide",
    "FG: N=C=S": "Isothiocyanate",
    "Atom count: S >= 1": "S atom count",
    "Atom count: S >= 2": "S atom count",
    "KG_Ancestor__AromaticMonocyclicCarbocycle": "Aromatic carbocycle",
    "KG_Ancestor__AromaticMonocyclicSulfurHeterocycle": "Aromatic S-heterocycle",
    "FG: Halogen-containing motif (token: F-Cl-Br-I)": "Halogen-containing motif",
    "FG: Ester(–COOR)": "Ester (-COOR)",
}


def normalize_feature_key(s):
    s = str(s)
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "", s)
    return s.lower()


FEATURE_LABEL_MAP_NORM = {
    normalize_feature_key(k): v for k, v in FEATURE_LABEL_MAP.items()
}


def pretty_feature_label(feature_name, axis="x"):
    raw = str(feature_name)

    if raw in FEATURE_LABEL_MAP:
        return FEATURE_LABEL_MAP[raw]

    norm = normalize_feature_key(raw)
    if norm in FEATURE_LABEL_MAP_NORM:
        return FEATURE_LABEL_MAP_NORM[norm]

    s = raw
    s = s.replace("KG_Ancestor__", "")
    s = s.replace("KG_FG__", "")
    s = s.replace("FG: ", "")
    s = s.replace("GroupsContainingSulfur", "S-containing groups")
    s = s.replace("GroupsContainingNitrogen", "N-containing groups")
    s = s.replace("NumAliphaticCarbons", "Aliphatic C")
    s = s.replace("MolWt", "MW")
    s = s.replace("&&", " && ")
    s = s.replace(">=", "≥")
    s = s.replace("<=", "≤")

    width = 28 if axis == "y" else 32

    if len(s) > width:
        s = "\n".join(
            textwrap.wrap(
                s,
                width=width,
                break_long_words=False,
                break_on_hyphens=False
            )
        )

    return s


def setup_matplotlib_style():
    available_fonts = {f.name for f in fm.fontManager.ttflist}

    if "Arial" in available_fonts:
        font_family = "Arial"
    else:
        font_family = "DejaVu Sans"
        print("[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.")

    mpl.rcParams.update({
        "font.family": font_family,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "axes.linewidth": AXIS_LINEWIDTH,
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.unicode_minus": False,
    })


setup_matplotlib_style()


def style_axis(ax):
    ax.tick_params(
        axis="both",
        which="major",
        width=TICK_WIDTH,
        length=TICK_LENGTH,
        labelsize=FONT_TICK
    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(AXIS_LINEWIDTH)


def style_colorbar(cbar, label, label_fontsize):
    cbar.set_label(
        label,
        fontsize=label_fontsize,
        fontweight="bold",
        labelpad=12
    )

    cbar.ax.tick_params(
        labelsize=CBAR_TICK_FONT,
        width=TICK_WIDTH,
        length=TICK_LENGTH
    )

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")


# ============================================================
# 6. 通用工具
# ============================================================

def normalize_name(s):
    s = str(s)
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "", s)
    return s.lower()


def resolve_feature_name(requested, columns):
    if requested in columns:
        return requested

    nr = normalize_name(requested)
    matches = [c for c in columns if normalize_name(c) == nr]

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        print(f"[WARN] Multiple matches for {requested}; use {matches[0]}")
        return matches[0]

    return None


def sanitize_filename(s):
    return re.sub(r"[^\w\-_\.]+", "_", str(s))


def find_smiles_col(df):
    candidates = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]

    if not candidates:
        return None

    priority = [
        "Canonical SMILES", "Canonical_SMILES", "canonical_smiles",
        "SMILES", "smiles", "StdSMILES"
    ]

    for p in priority:
        for c in candidates:
            if c.lower() == p.lower():
                return c

    return candidates[0]


def is_numeric_or_convertible(series):
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def is_interpretable_feature(feature):
    f = str(feature).lower()

    if f.startswith("morgan_"):
        return False

    if f.startswith("kg_rel"):
        return False

    allowed_keywords = [
        "fg:", "kg_fg", "kg_ancestor", "kg_scaffold",
        "atom", "count", "exact", "molwt", "num",
        "c(=o)", "[cx3]", "n=c=s", "cooh",
        "thiol", "amine", "aldehyde", "sulf",
        "ether", "ester", "aromatic", "furan", "thiophene",
        "halogen", "chloro", "bromo", "fluoro"
    ]

    return any(k in f for k in allowed_keywords)


def build_X_y(df):
    smiles_col = find_smiles_col(df)

    missing = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if missing:
        raise ValueError(f"缺少标签列: {missing}")

    y_df = df[TARGET_LABELS_24].fillna(0).astype(int)

    exclude = set(TARGET_LABELS_24)

    if smiles_col is not None:
        exclude.add(smiles_col)

    feature_cols = [c for c in df.columns if c not in exclude]

    good_cols = []
    bad_cols = []

    for c in feature_cols:
        if is_numeric_or_convertible(df[c]):
            good_cols.append(c)
        else:
            bad_cols.append(c)

    if bad_cols:
        print(f"[WARN] 删除非数值特征列 {len(bad_cols)} 个")
        print(bad_cols[:20])

    X_df = df[good_cols].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X_df = X_df.fillna(0).astype(np.float32)

    return X_df, y_df, smiles_col


# ============================================================
# 7. 特征分组
# ============================================================

def match_group(feature_name, group_name):
    f = str(feature_name).lower()
    return any(re.search(pat, f) for pat in GROUP_PATTERNS[group_name])


def assign_groups(feature_name):
    return [g for g in GROUP_PATTERNS if match_group(feature_name, g)]


def export_detected_group_features(feature_names):
    rows = []

    for i, f in enumerate(feature_names):
        groups = assign_groups(f)

        if groups and is_interpretable_feature(f):
            rows.append({
                "feature_index": i,
                "feature": f,
                "matched_groups": ";".join(groups)
            })

    out = pd.DataFrame(rows)

    out_path = os.path.join(TABLE_DIR, "detected_interpretable_group_features.csv")
    out.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("[SAVE]", out_path)

    return out


# ============================================================
# 8. 模型训练与预测
# ============================================================

def train_xgb_binary(X_values, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    model = xgb.XGBClassifier(**params)
    model.fit(X_values, y_bin)

    return model


def predict_proba(model, X_df):
    return model.predict_proba(X_df.values)[:, 1]


def get_xgb_gain_importance(model, feature_names):
    score = model.get_booster().get_score(importance_type="gain")
    importance = np.zeros(len(feature_names), dtype=float)

    for k, v in score.items():
        if k.startswith("f"):
            idx = int(k[1:])
            if 0 <= idx < len(feature_names):
                importance[idx] = float(v)

    return importance


# ============================================================
# 9. 支持度与冗余
# ============================================================

def is_binary_feature(values):
    vals = pd.Series(values).dropna().unique()
    vals = set(float(v) for v in vals)
    return vals.issubset({0.0, 1.0})


def non_constant_feature(X_df, feature):
    return np.nanstd(X_df[feature].values.astype(float)) > 1e-12


def feature_support_ok(X_df, feature):
    if not non_constant_feature(X_df, feature):
        return False, "constant_feature"

    x = X_df[feature].values.astype(float)

    if is_binary_feature(x):
        n1 = int(np.sum(x > 0))
        n0 = int(np.sum(x <= 0))

        if n1 < MIN_FEATURE_SUPPORT_BINARY:
            return False, f"binary_positive_support_too_low_n1={n1}"

        if n0 < MIN_FEATURE_SUPPORT_BINARY:
            return False, f"binary_negative_support_too_low_n0={n0}"

    return True, "ok"


def pair_support_ok(X_df, feature_a, feature_b):
    xa = X_df[feature_a].values.astype(float)
    xb = X_df[feature_b].values.astype(float)

    support = {
        "n00": np.nan,
        "n10": np.nan,
        "n01": np.nan,
        "n11": np.nan,
        "pair_support_status": "not_binary_pair"
    }

    if is_binary_feature(xa) and is_binary_feature(xb):
        a = (xa > 0).astype(int)
        b = (xb > 0).astype(int)

        n00 = int(np.sum((a == 0) & (b == 0)))
        n10 = int(np.sum((a == 1) & (b == 0)))
        n01 = int(np.sum((a == 0) & (b == 1)))
        n11 = int(np.sum((a == 1) & (b == 1)))

        support.update({
            "n00": n00,
            "n10": n10,
            "n01": n01,
            "n11": n11,
            "pair_support_status": "binary_pair"
        })

        if n11 < MIN_PAIR_COOC_SUPPORT_BINARY:
            return False, f"cooccurrence_too_low_n11={n11}", support

    return True, "ok", support


def binary_redundancy_stats(X_df, feature_a, feature_b):
    xa = X_df[feature_a].values.astype(float)
    xb = X_df[feature_b].values.astype(float)

    out = {
        "binary_phi_corr": np.nan,
        "binary_jaccard": np.nan,
        "redundant_pair": False,
        "redundancy_reason": "ok"
    }

    if not (is_binary_feature(xa) and is_binary_feature(xb)):
        return out

    a = (xa > 0).astype(int)
    b = (xb > 0).astype(int)

    if np.std(a) == 0 or np.std(b) == 0:
        out["redundant_pair"] = True
        out["redundancy_reason"] = "binary_constant"
        return out

    phi = float(np.corrcoef(a, b)[0, 1])

    intersection = np.sum((a == 1) & (b == 1))
    union = np.sum((a == 1) | (b == 1))
    jaccard = float(intersection / union) if union > 0 else np.nan

    out["binary_phi_corr"] = phi
    out["binary_jaccard"] = jaccard

    if np.isfinite(phi) and abs(phi) >= MAX_BINARY_PHI_CORR:
        out["redundant_pair"] = True
        out["redundancy_reason"] = f"high_phi_corr={phi:.3f}"

    if np.isfinite(jaccard) and jaccard >= MAX_BINARY_JACCARD:
        out["redundant_pair"] = True
        out["redundancy_reason"] = f"high_jaccard={jaccard:.3f}"

    return out


# ============================================================
# 10. PDP / ICE 计算
# ============================================================

def sample_X(X_df, max_n, random_seed=42):
    if len(X_df) <= max_n:
        return X_df.copy()

    rng = np.random.default_rng(random_seed)
    idx = rng.choice(np.arange(len(X_df)), size=max_n, replace=False)

    return X_df.iloc[idx].copy()


def make_grid(values, n_grid=N_GRID_CONTINUOUS, max_unique_as_discrete=MAX_UNIQUE_AS_DISCRETE):
    values = pd.Series(values).dropna().astype(float)
    unique_vals = np.sort(values.unique())

    if len(unique_vals) <= max_unique_as_discrete:
        return unique_vals

    qs = np.linspace(0.05, 0.95, n_grid)
    grid = np.quantile(values, qs)
    grid = np.unique(np.round(grid, 6))

    return grid


def compute_1d_pdp(model, X_ref, feature, grid):
    vals = []

    for v in grid:
        X_tmp = X_ref.copy()
        X_tmp[feature] = v
        vals.append(float(np.mean(predict_proba(model, X_tmp))))

    return np.array(vals)


def compute_2d_pdp(model, X_ref, feature_a, grid_a, feature_b, grid_b):
    mat = np.zeros((len(grid_b), len(grid_a)), dtype=float)

    for i, vb in enumerate(grid_b):
        for j, va in enumerate(grid_a):
            X_tmp = X_ref.copy()
            X_tmp[feature_a] = va
            X_tmp[feature_b] = vb
            mat[i, j] = float(np.mean(predict_proba(model, X_tmp)))

    return mat


def compute_interaction_residual(pdp2d, pdp_a, pdp_b, baseline):
    residual = np.zeros_like(pdp2d)

    for i in range(len(pdp_b)):
        for j in range(len(pdp_a)):
            residual[i, j] = pdp2d[i, j] - pdp_a[j] - pdp_b[i] + baseline

    return residual


def compute_ice_curves(model, X_ref, feature_a, grid_a, feature_b, b_levels):
    out = {}

    for vb in b_levels:
        curves = []

        for va in grid_a:
            X_tmp = X_ref.copy()
            X_tmp[feature_a] = va
            X_tmp[feature_b] = vb
            curves.append(predict_proba(model, X_tmp))

        out[vb] = np.vstack(curves).T

    return out


# ============================================================
# 11. 交互判断
# ============================================================

def judge_interaction(metrics):
    if metrics.get("redundant_pair", False):
        return False, "redundant_features", metrics.get("redundancy_reason", "redundant_pair")

    pdp_range = metrics["pdp_range"]
    mean_abs = metrics["mean_abs_interaction_residual"]
    max_abs = metrics["max_abs_interaction_residual"]
    contrast = metrics["interaction_contrast_2x2"]

    reasons = []

    if pdp_range < MIN_PDP_RANGE:
        reasons.append(f"pdp_range_too_small={pdp_range:.4f}")

    residual_signal = (
        mean_abs >= MIN_MEAN_ABS_RESIDUAL
    ) or (
        max_abs >= MIN_MAX_ABS_RESIDUAL
    )

    contrast_signal = (
        np.isfinite(contrast)
        and abs(contrast) >= MIN_BINARY_INTERACTION_CONTRAST
    )

    if not residual_signal and not contrast_signal:
        reasons.append(
            f"interaction_too_weak: "
            f"mean_abs={mean_abs:.4f}, "
            f"max_abs={max_abs:.4f}, "
            f"contrast={contrast:.4f}"
        )

    if reasons:
        return False, "no_clear_interaction", "; ".join(reasons)

    signal = contrast if np.isfinite(contrast) else metrics["signed_mean_interaction_residual"]

    if signal > 0:
        return True, "synergistic_positive", "positive interaction signal"
    elif signal < 0:
        return True, "antagonistic_negative", "negative interaction signal"
    else:
        return True, "interaction_strength_only", "non-directional interaction strength"


def compute_pair_result(model, X_df, label, feature_a, feature_b, max_samples):
    ok_a, reason_a = feature_support_ok(X_df, feature_a)

    if not ok_a:
        return None, f"feature_a_failed: {reason_a}"

    ok_b, reason_b = feature_support_ok(X_df, feature_b)

    if not ok_b:
        return None, f"feature_b_failed: {reason_b}"

    ok_pair, pair_reason, support = pair_support_ok(X_df, feature_a, feature_b)

    if not ok_pair:
        return None, f"pair_failed: {pair_reason}"

    redundancy = binary_redundancy_stats(X_df, feature_a, feature_b)

    X_ref = sample_X(X_df, max_samples, RANDOM_SEED)

    grid_a = make_grid(X_ref[feature_a].values)
    grid_b = make_grid(X_ref[feature_b].values)

    if len(grid_a) < 2 or len(grid_b) < 2:
        return None, "grid_too_small"

    baseline = float(np.mean(predict_proba(model, X_ref)))

    pdp_a = compute_1d_pdp(model, X_ref, feature_a, grid_a)
    pdp_b = compute_1d_pdp(model, X_ref, feature_b, grid_b)
    pdp2d = compute_2d_pdp(model, X_ref, feature_a, grid_a, feature_b, grid_b)
    residual = compute_interaction_residual(pdp2d, pdp_a, pdp_b, baseline)

    interaction_contrast = np.nan
    interaction_direction = "not_2x2"

    if len(grid_a) == 2 and len(grid_b) == 2:
        try:
            ia0 = int(np.where(grid_a == 0)[0][0])
            ia1 = int(np.where(grid_a == 1)[0][0])
            ib0 = int(np.where(grid_b == 0)[0][0])
            ib1 = int(np.where(grid_b == 1)[0][0])

            f00 = float(pdp2d[ib0, ia0])
            f10 = float(pdp2d[ib0, ia1])
            f01 = float(pdp2d[ib1, ia0])
            f11 = float(pdp2d[ib1, ia1])

            interaction_contrast = f11 - f10 - f01 + f00

            if interaction_contrast > 0:
                interaction_direction = "synergistic_positive"
            elif interaction_contrast < 0:
                interaction_direction = "antagonistic_negative"
            else:
                interaction_direction = "near_zero"

        except Exception:
            pass

    metrics = {
        "label": label,
        "feature_a": feature_a,
        "feature_b": feature_b,
        "baseline_probability": baseline,
        "grid_a_size": len(grid_a),
        "grid_b_size": len(grid_b),
        "grid_a": ";".join(map(str, grid_a)),
        "grid_b": ";".join(map(str, grid_b)),
        "min_PDP2D": float(np.min(pdp2d)),
        "max_PDP2D": float(np.max(pdp2d)),
        "pdp_range": float(np.max(pdp2d) - np.min(pdp2d)),
        "min_interaction_residual": float(np.min(residual)),
        "max_interaction_residual": float(np.max(residual)),
        "mean_abs_interaction_residual": float(np.mean(np.abs(residual))),
        "max_abs_interaction_residual": float(np.max(np.abs(residual))),
        "signed_mean_interaction_residual": float(np.mean(residual)),
        "interaction_contrast_2x2": float(interaction_contrast) if np.isfinite(interaction_contrast) else np.nan,
        "interaction_direction_2x2": interaction_direction,
    }

    metrics.update(support)
    metrics.update(redundancy)

    useful, conclusion, reason = judge_interaction(metrics)

    metrics["useful_for_plot"] = useful
    metrics["conclusion_type"] = conclusion
    metrics["decision_reason"] = reason

    return {
        "metrics": metrics,
        "grid_a": grid_a,
        "grid_b": grid_b,
        "pdp2d": pdp2d,
        "residual": residual,
    }, "ok"


# ============================================================
# 12. 候选 pair 筛选
# ============================================================

def candidate_features_by_group(group, X_df, feature_names, importance):
    rows = []

    for idx, f in enumerate(feature_names):
        if not is_interpretable_feature(f):
            continue

        if not match_group(f, group):
            continue

        ok, reason = feature_support_ok(X_df, f)

        if not ok:
            continue

        rows.append({
            "group": group,
            "feature_index": idx,
            "feature": f,
            "gain_importance": float(importance[idx]),
            "support_reason": reason,
            "matched_groups": ";".join(assign_groups(f)),
        })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values("gain_importance", ascending=False).head(TOPK_FEATURES_PER_GROUP)


def get_manual_candidate_pairs(label, X_df, feature_names, importance):
    rows = []

    if label not in MANUAL_PAIRS:
        return rows

    for fa_raw, fb_raw in MANUAL_PAIRS[label]:
        fa = resolve_feature_name(fa_raw, X_df.columns)
        fb = resolve_feature_name(fb_raw, X_df.columns)

        if fa is None:
            print(f"[WARN] Manual feature A not found: {fa_raw}")
            continue

        if fb is None:
            print(f"[WARN] Manual feature B not found: {fb_raw}")
            continue

        ia = feature_names.index(fa)
        ib = feature_names.index(fb)

        rows.append({
            "label": label,
            "group_a": assign_groups(fa)[0] if assign_groups(fa) else "Manual",
            "feature_a": fa,
            "importance_a": float(importance[ia]),
            "group_b": assign_groups(fb)[0] if assign_groups(fb) else "Manual",
            "feature_b": fb,
            "importance_b": float(importance[ib]),
            "selection_mode": "manual_curated"
        })

    return rows


def forced_score(row):
    pdp_range = row.get("pdp_range", 0.0)
    mean_abs = row.get("mean_abs_interaction_residual", 0.0)
    max_abs = row.get("max_abs_interaction_residual", 0.0)
    contrast = row.get("interaction_contrast_2x2", 0.0)

    if not np.isfinite(pdp_range):
        pdp_range = 0.0
    if not np.isfinite(mean_abs):
        mean_abs = 0.0
    if not np.isfinite(max_abs):
        max_abs = 0.0
    if not np.isfinite(contrast):
        contrast = 0.0

    return 1.0 * pdp_range + 1.5 * max_abs + 1.0 * mean_abs + 1.0 * abs(contrast)


def screen_candidate_pairs(label, model, X_df, feature_names, importance):
    rows = []
    used_pairs = set()

    # 1. 人工候选 pair 优先
    for r in get_manual_candidate_pairs(label, X_df, feature_names, importance):
        key = tuple(sorted([r["feature_a"], r["feature_b"]]))

        if key not in used_pairs:
            rows.append(r)
            used_pairs.add(key)

    # 2. 自动候选 pair
    plan = LABEL_GROUP_PAIR_PLAN.get(label, [])

    for group_a, group_b in plan:
        cand_a = candidate_features_by_group(group_a, X_df, feature_names, importance)
        cand_b = candidate_features_by_group(group_b, X_df, feature_names, importance)

        if cand_a.empty or cand_b.empty:
            continue

        count = 0

        for _, ra in cand_a.iterrows():
            for _, rb in cand_b.iterrows():
                fa = ra["feature"]
                fb = rb["feature"]

                if fa == fb:
                    continue

                key = tuple(sorted([fa, fb]))

                if key in used_pairs:
                    continue

                rows.append({
                    "label": label,
                    "group_a": group_a,
                    "feature_a": fa,
                    "importance_a": float(ra["gain_importance"]),
                    "group_b": group_b,
                    "feature_b": fb,
                    "importance_b": float(rb["gain_importance"]),
                    "selection_mode": "auto_group_importance"
                })

                used_pairs.add(key)
                count += 1

                if count >= MAX_PAIRS_PER_GROUP_COMBO:
                    break

            if count >= MAX_PAIRS_PER_GROUP_COMBO:
                break

    if not rows:
        return pd.DataFrame(), pd.DataFrame()

    evaluated = []

    for r in rows:
        result, status = compute_pair_result(
            model=model,
            X_df=X_df,
            label=label,
            feature_a=r["feature_a"],
            feature_b=r["feature_b"],
            max_samples=MAX_SCREEN_PDP_SAMPLES
        )

        base = dict(r)
        base["screen_status"] = status

        if result is None:
            base["useful_for_plot"] = False
            base["conclusion_type"] = "failed_before_metrics"
            base["decision_reason"] = status
            base["forced_score"] = 0.0
            evaluated.append(base)
            continue

        base.update(result["metrics"])
        base["forced_score"] = forced_score(base)
        evaluated.append(base)

    eval_df = pd.DataFrame(evaluated)

    all_path = os.path.join(TABLE_DIR, f"ALL_candidate_pair_decision__{label}.csv")
    eval_df.to_csv(all_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", all_path)

    # 3. 先选 useful pair
    useful_df = eval_df[eval_df["useful_for_plot"] == True].copy()

    selected_rows = []

    if not useful_df.empty:
        useful_df = useful_df.sort_values(
            ["mean_abs_interaction_residual", "max_abs_interaction_residual", "pdp_range"],
            ascending=False
        )

        for _, row in useful_df.iterrows():
            selected_rows.append(row)

            if len(selected_rows) >= MAX_PLOTS_PER_LABEL:
                break

    # 4. 如果 useful 不足，则强制补足
    if FORCE_PLOT_SUBJECTIVE_MALODORS and label in SUBJECTIVE_MALODOR_LABELS:
        selected_keys = set(
            tuple(sorted([r["feature_a"], r["feature_b"]]))
            for r in selected_rows
        )

        valid_df = eval_df[
            ~eval_df["conclusion_type"].isin(["failed_before_metrics"])
        ].copy()

        if not valid_df.empty:
            valid_df = valid_df.sort_values(
                ["selection_mode", "forced_score"],
                ascending=[True, False]
            )

            for _, row in valid_df.iterrows():
                key = tuple(sorted([row["feature_a"], row["feature_b"]]))

                if key in selected_keys:
                    continue

                row = row.copy()
                row["force_plot"] = True
                row["force_plot_reason"] = "subjective_malodor_label_forced_representative_plot"
                selected_rows.append(row)
                selected_keys.add(key)

                if len(selected_rows) >= max(MIN_PLOTS_PER_LABEL, MAX_PLOTS_PER_LABEL):
                    break

    if len(selected_rows) == 0:
        return eval_df, pd.DataFrame()

    selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)
    selected_df.insert(1, "plot_pair_id", np.arange(1, len(selected_df) + 1))

    selected_path = os.path.join(TABLE_DIR, f"SELECTED_pairs_for_plot__{label}.csv")
    selected_df.to_csv(selected_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", selected_path)

    return eval_df, selected_df


# ============================================================
# 13. 绘图函数
# ============================================================

def save_figure(fig, out_png):
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    print("[SAVE]", out_png)

    if SAVE_PDF:
        out_pdf = out_png.replace(".png", ".pdf")
        fig.savefig(out_pdf, bbox_inches="tight")
        print("[SAVE]", out_pdf)

    if SAVE_SVG:
        out_svg = out_png.replace(".png", ".svg")
        fig.savefig(out_svg, bbox_inches="tight")
        print("[SAVE]", out_svg)


def is_discrete_grid(grid):
    return len(grid) <= MAX_UNIQUE_AS_DISCRETE


def format_grid_labels(grid):
    labels = []

    for v in grid:
        try:
            vf = float(v)
            labels.append(str(int(vf)) if vf.is_integer() else f"{vf:.3g}")
        except Exception:
            labels.append(str(v))

    return labels


def plot_discrete_heatmap_from_matrix(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    cbar_label,
    out_png,
    signed=False,
    cbar_label_fontsize=22
):
    mat = np.asarray(matrix, dtype=float)

    if signed and RESIDUAL_COLOR_BY_ABS:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        colorbar_label = f"|{cbar_label}|" if ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR else cbar_label

    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0

        colorbar_label = cbar_label

    fig_w = max(6.6, 1.55 * len(grid_a) + 3.6)
    fig_h = max(5.4, 1.15 * len(grid_b) + 2.8)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        color_mat,
        origin="lower",
        aspect="auto",
        cmap=BLUE_WHITE_CMAP,
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(im, ax=ax)
    style_colorbar(cbar, colorbar_label, cbar_label_fontsize)

    ax.set_xticks(np.arange(len(grid_a)))
    ax.set_yticks(np.arange(len(grid_b)))

    ax.set_xticklabels(format_grid_labels(grid_a), fontweight="bold")
    ax.set_yticklabels(format_grid_labels(grid_b), fontweight="bold")

    ax.set_xlabel(
        pretty_feature_label(feature_a, axis="x"),
        fontsize=FONT_X_LABEL,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        pretty_feature_label(feature_b, axis="y"),
        fontsize=FONT_Y_LABEL,
        fontweight="bold",
        labelpad=12
    )

    ax.set_title("")

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            raw_val = mat[i, j]
            color_val = color_mat[i, j]

            if not np.isfinite(raw_val):
                continue

            text = f"{raw_val:+.3f}" if signed else f"{raw_val:.3f}"

            if vmax > vmin:
                threshold = vmin + 0.62 * (vmax - vmin)
                text_color = "white" if color_val > threshold else "black"
            else:
                text_color = "black"

            ax.text(
                j, i, text,
                ha="center",
                va="center",
                fontsize=FONT_ANNOT,
                fontweight="bold",
                color=text_color
            )

    ax.set_xticks(np.arange(-0.5, len(grid_a), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(grid_b), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    style_axis(ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.20, right=0.88)

    save_figure(fig, out_png)
    plt.close(fig)


def plot_contour_from_matrix(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    cbar_label,
    out_png,
    signed=False,
    cbar_label_fontsize=22
):
    mat = np.asarray(matrix, dtype=float)
    Xg, Yg = np.meshgrid(grid_a, grid_b)

    if signed and RESIDUAL_COLOR_BY_ABS:
        color_mat = np.abs(mat)
        vmin = 0.0
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        levels = np.linspace(vmin, vmax, 24)
        colorbar_label = f"|{cbar_label}|" if ADD_ABS_SYMBOL_TO_RESIDUAL_CBAR else cbar_label

    else:
        color_mat = mat
        vmin = float(np.nanmin(color_mat))
        vmax = float(np.nanmax(color_mat))

        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0

        levels = np.linspace(vmin, vmax, 24)
        colorbar_label = cbar_label

    fig, ax = plt.subplots(figsize=(7.8, 6.3))

    cf = ax.contourf(
        Xg, Yg, color_mat,
        levels=levels,
        cmap=BLUE_WHITE_CMAP,
        extend="both"
    )

    cbar = plt.colorbar(cf, ax=ax)
    style_colorbar(cbar, colorbar_label, cbar_label_fontsize)

    try:
        cs = ax.contour(
            Xg, Yg, mat,
            levels=8,
            colors="#08306b",
            linewidths=1.3,
            alpha=0.85
        )

        ax.clabel(
            cs,
            inline=True,
            fontsize=FONT_ANNOT - 4,
            fmt="%.3f",
            colors="#08306b"
        )
    except Exception:
        pass

    ax.set_xlabel(
        pretty_feature_label(feature_a, axis="x"),
        fontsize=FONT_X_LABEL,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        pretty_feature_label(feature_b, axis="y"),
        fontsize=FONT_Y_LABEL,
        fontweight="bold",
        labelpad=12
    )

    ax.set_title("")

    style_axis(ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.22, right=0.88)

    save_figure(fig, out_png)
    plt.close(fig)


def plot_pdp2d_auto(
    matrix,
    grid_a,
    grid_b,
    feature_a,
    feature_b,
    cbar_label,
    out_png,
    signed=False,
    cbar_label_fontsize=22
):
    if is_discrete_grid(grid_a) or is_discrete_grid(grid_b):
        plot_discrete_heatmap_from_matrix(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed,
            cbar_label_fontsize=cbar_label_fontsize
        )
    else:
        plot_contour_from_matrix(
            matrix=matrix,
            grid_a=grid_a,
            grid_b=grid_b,
            feature_a=feature_a,
            feature_b=feature_b,
            cbar_label=cbar_label,
            out_png=out_png,
            signed=signed,
            cbar_label_fontsize=cbar_label_fontsize
        )


def plot_ice_curves(
    model,
    X_df,
    label,
    feature_a,
    feature_b,
    grid_a,
    grid_b,
    out_png
):
    X_ice = sample_X(X_df, MAX_ICE_SAMPLES, RANDOM_SEED)

    if len(grid_b) > 3:
        b_levels = [grid_b[0], grid_b[len(grid_b) // 2], grid_b[-1]]
    else:
        b_levels = list(grid_b)

    ice = compute_ice_curves(
        model=model,
        X_ref=X_ice,
        feature_a=feature_a,
        grid_a=grid_a,
        feature_b=feature_b,
        b_levels=b_levels
    )

    fig, ax = plt.subplots(figsize=(7.8, 6.0))
    rng = np.random.default_rng(RANDOM_SEED)

    for idx_level, (vb, curves) in enumerate(ice.items()):
        n = curves.shape[0]

        if n > MAX_ICE_LINES_PER_LEVEL:
            idx = rng.choice(np.arange(n), size=MAX_ICE_LINES_PER_LEVEL, replace=False)
        else:
            idx = np.arange(n)

        for k in idx:
            ax.plot(
                grid_a, curves[k],
                color=ICE_INDIVIDUAL_COLOR,
                alpha=0.16,
                linewidth=ICE_LINE_WIDTH
            )

        mean_curve = curves.mean(axis=0)
        mean_color = ICE_MEAN_COLORS[idx_level % len(ICE_MEAN_COLORS)]

        ax.plot(
            grid_a, mean_curve,
            color=mean_color,
            linewidth=MEAN_LINE_WIDTH,
            marker="o",
            markersize=8,
            markeredgewidth=1.5,
            label=f"{pretty_feature_label(feature_b, axis='y')}={vb}"
        )

    ax.set_xlabel(
        pretty_feature_label(feature_a, axis="x"),
        fontsize=FONT_X_LABEL,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        f"Predicted probability of {label}",
        fontsize=FONT_Y_LABEL,
        fontweight="bold",
        labelpad=12
    )

    ax.set_title("")

    legend = ax.legend(frameon=False, fontsize=14)

    for text in legend.get_texts():
        text.set_fontweight("bold")

    style_axis(ax)

    plt.tight_layout()
    save_figure(fig, out_png)
    plt.close(fig)


def save_pair_matrices(
    label,
    pair_id,
    feature_a,
    feature_b,
    grid_a,
    grid_b,
    pdp2d,
    residual
):
    prefix = (
        f"{label}__pair{pair_id:02d}__"
        f"{sanitize_filename(feature_a)}__x__{sanitize_filename(feature_b)}"
    )

    pd.DataFrame(
        pdp2d,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_probability_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        residual,
        index=[f"{feature_b}={v}" for v in grid_b],
        columns=[f"{feature_a}={v}" for v in grid_a]
    ).to_csv(
        os.path.join(TABLE_DIR, f"PDP2D_interaction_residual_matrix__{prefix}.csv"),
        encoding="utf-8-sig"
    )


def plot_final_pair(model, X_df, label, pair_id, pair_row):
    feature_a = pair_row["feature_a"]
    feature_b = pair_row["feature_b"]

    result, status = compute_pair_result(
        model=model,
        X_df=X_df,
        label=label,
        feature_a=feature_a,
        feature_b=feature_b,
        max_samples=MAX_FINAL_PDP_SAMPLES
    )

    if result is None:
        print(f"[SKIP FINAL] {label}: {feature_a} × {feature_b}: {status}")
        return None

    metrics = result["metrics"]
    grid_a = result["grid_a"]
    grid_b = result["grid_b"]
    pdp2d = result["pdp2d"]
    residual = result["residual"]

    # 主观恶臭标签强制出图
    if FORCE_PLOT_SUBJECTIVE_MALODORS and label in SUBJECTIVE_MALODOR_LABELS:
        metrics["force_plot"] = True
    else:
        metrics["force_plot"] = False

    save_pair_matrices(
        label=label,
        pair_id=pair_id,
        feature_a=feature_a,
        feature_b=feature_b,
        grid_a=grid_a,
        grid_b=grid_b,
        pdp2d=pdp2d,
        residual=residual
    )

    if (not metrics["useful_for_plot"]) and (not metrics["force_plot"]):
        print(f"[NO PLOT FINAL] {label}: {feature_a} × {feature_b}: {metrics['decision_reason']}")
        return metrics

    prefix = (
        f"{label}__pair{pair_id:02d}__"
        f"{sanitize_filename(feature_a)}__x__{sanitize_filename(feature_b)}"
    )

    plot_pdp2d_auto(
        matrix=pdp2d,
        grid_a=grid_a,
        grid_b=grid_b,
        feature_a=feature_a,
        feature_b=feature_b,
        cbar_label="Partial dependence/predicted probability",
        out_png=os.path.join(PLOT_DIR, f"PDP2D_probability__{prefix}.png"),
        signed=False,
        cbar_label_fontsize=CBAR_LABEL_FONT_PROB
    )

    plot_pdp2d_auto(
        matrix=residual,
        grid_a=grid_a,
        grid_b=grid_b,
        feature_a=feature_a,
        feature_b=feature_b,
        cbar_label="PDP interaction residual",
        out_png=os.path.join(PLOT_DIR, f"PDP2D_interaction_residual__{prefix}.png"),
        signed=True,
        cbar_label_fontsize=CBAR_LABEL_FONT_RESID
    )

    if PLOT_ICE:
        plot_ice_curves(
            model=model,
            X_df=X_df,
            label=label,
            feature_a=feature_a,
            feature_b=feature_b,
            grid_a=grid_a,
            grid_b=grid_b,
            out_png=os.path.join(PLOT_DIR, f"ICE_curves__{prefix}.png")
        )

    return metrics


# ============================================================
# 14. 单标签流程
# ============================================================

def analyze_one_label(label, X_df, y_df, feature_names):
    print("\n" + "=" * 100)
    print(f"[LABEL] {label}")
    print("=" * 100)

    y_bin = y_df[label].values.astype(int)
    n_pos = int(y_bin.sum())
    n_neg = int(len(y_bin) - n_pos)

    print(f"[INFO] positives={n_pos}, negatives={n_neg}")

    if n_pos < MIN_POSITIVES_PER_LABEL:
        print(f"[SKIP] {label}: positives < {MIN_POSITIVES_PER_LABEL}")
        return None

    model = train_xgb_binary(X_df.values, y_bin)

    model_path = os.path.join(MODEL_DIR, f"xgb__{label}.json")
    model.get_booster().save_model(model_path)
    print("[SAVE]", model_path)

    importance = get_xgb_gain_importance(model, feature_names)

    imp_df = pd.DataFrame({
        "feature": feature_names,
        "gain_importance": importance,
        "matched_groups": [";".join(assign_groups(f)) for f in feature_names],
        "interpretable_for_interaction": [is_interpretable_feature(f) for f in feature_names]
    }).sort_values("gain_importance", ascending=False)

    imp_path = os.path.join(TABLE_DIR, f"XGB_gain_importance__{label}.csv")
    imp_df.to_csv(imp_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", imp_path)

    all_pairs_df, selected_pairs_df = screen_candidate_pairs(
        label=label,
        model=model,
        X_df=X_df,
        feature_names=feature_names,
        importance=importance
    )

    if selected_pairs_df.empty:
        print(f"[NO SELECTED PAIR] {label}: no valid pair can be plotted")
        return pd.DataFrame()

    final_metrics = []

    for _, row in selected_pairs_df.iterrows():
        pair_id = int(row["plot_pair_id"])

        metrics = plot_final_pair(
            model=model,
            X_df=X_df,
            label=label,
            pair_id=pair_id,
            pair_row=row
        )

        if metrics is not None:
            metrics["plot_pair_id"] = pair_id
            metrics["selected_feature_a"] = row["feature_a"]
            metrics["selected_feature_b"] = row["feature_b"]
            metrics["selection_mode"] = row.get("selection_mode", "")
            metrics["selected_from_useful"] = bool(row.get("useful_for_plot", False))
            final_metrics.append(metrics)

    if final_metrics:
        final_df = pd.DataFrame(final_metrics)

        out_path = os.path.join(TABLE_DIR, f"FINAL_plotted_pair_metrics__{label}.csv")
        final_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("[SAVE]", out_path)

        return final_df

    return pd.DataFrame()


# ============================================================
# 15. 主程序
# ============================================================

def main():
    print("[INFO] Reading feature file:")
    print(FEATURE_FILE)

    df = pd.read_excel(FEATURE_FILE, sheet_name=SHEET_NAME)

    X_df, y_df, smiles_col = build_X_y(df)
    feature_names = X_df.columns.tolist()

    print("[INFO] X shape:", X_df.shape)
    print("[INFO] y shape:", y_df.shape)
    print("[INFO] n_features:", len(feature_names))

    if smiles_col is not None:
        print("[INFO] SMILES column:", smiles_col)

    export_detected_group_features(feature_names)

    meta = {
        "feature_file": FEATURE_FILE,
        "method": "Forced representative PDP/ICE plots for subjectively malodorous descriptors",
        "labels": ANALYSIS_LABELS,
        "subjective_malodor_labels": SUBJECTIVE_MALODOR_LABELS,
        "force_plot_subjective_malodors": FORCE_PLOT_SUBJECTIVE_MALODORS,
        "min_plots_per_label": MIN_PLOTS_PER_LABEL,
        "max_plots_per_label": MAX_PLOTS_PER_LABEL,
        "plot_policy": {
            "png_only": True,
            "dpi": DPI,
            "no_pdf": True,
            "no_title": True,
            "blue_white_colormap": True,
            "discrete_or_binary_features": "block heatmap",
            "continuous_features": "contour plot",
            "ice_curves": PLOT_ICE
        },
        "interpretation_note": (
            "Forced plots are representative model-response visualizations. "
            "If useful_for_plot=False, the figure should not be interpreted as strong evidence "
            "of interaction; it can be used as supplementary evidence or to show weak/no interaction."
        )
    }

    with open(os.path.join(OUT_DIR, "analysis_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    all_final = []

    for label in ANALYSIS_LABELS:
        if label not in y_df.columns:
            print(f"[WARN] label not found: {label}")
            continue

        result = analyze_one_label(label, X_df, y_df, feature_names)

        if result is not None and not result.empty:
            all_final.append(result)

    if all_final:
        all_df = pd.concat(all_final, axis=0, ignore_index=True)

        out_csv = os.path.join(TABLE_DIR, "ALL_FINAL_plotted_pair_metrics.csv")
        out_xlsx = os.path.join(TABLE_DIR, "ALL_FINAL_plotted_pair_metrics.xlsx")

        all_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
        all_df.to_excel(out_xlsx, index=False)

        print("[SAVE]", out_csv)
        print("[SAVE]", out_xlsx)

    print("\n[DONE]")
    print("Plots:", PLOT_DIR)
    print("Tables:", TABLE_DIR)


if __name__ == "__main__":
    main()

[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.
[INFO] Reading feature file:
./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X shape: (3756, 2595)
[INFO] y shape: (3756, 24)
[INFO] n_features: 2595
[INFO] SMILES column: Canonical_SMILES
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/tables/detected_interpretable_group_features.csv

[LABEL] sulfurous
[INFO] positives=398, negatives=3358
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/models/xgb__sulfurous.json
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/tables/XGB_gain_importance__sulfurous.csv
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/tables/ALL_candidate_pair_decision__sulfurous.csv
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/tables/SELECTED_pairs_for_plot__sulfurous.csv
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/plots/PDP2D_probability__sulfurous__pair01__Atom_count_S_1__x__Atom_count_N_2.png
[SAVE] ./Forced_subjective_malodor_PDP_ICE_plots/plots/PDP2D_interaction_residual__sulf